# Synthetic dataset analysis

Reads `data/synthetic/metadata/frames.jsonl` (written by `scripts/generate_dataset.py`). Run from `vision/` with `uv run --with jupyter jupyter lab` (Jupyter is not a project dependency).

In [ ]:
import json, collections
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt, cv2
DATA = Path('../data/synthetic')
records = [json.loads(l) for l in (DATA/'metadata/frames.jsonl').read_text().splitlines() if l.strip()]
generation = json.loads((DATA/'metadata/generation.json').read_text())
print(len(records), 'images; renderer:', generation['renderer']['webgl'])

In [ ]:
objects = pd.DataFrame([{**o, 'split': r['split'], 'night': r['night'], 'robot': r['robot_id'], 'mode': r['view_mode'],
    'w': o['bbox_xyxy'][2]-o['bbox_xyxy'][0], 'h': o['bbox_xyxy'][3]-o['bbox_xyxy'][1]} for r in records for o in r['objects']])
objects.pivot_table(index='kind', columns='split', values='id', aggfunc='count')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for kind, g in objects.groupby('kind'):
    axes[0].hist((g.w*g.h)**.5, bins=40, histtype='step', label=kind)
    axes[1].hist(g.distance_m.dropna(), bins=30, range=(0, 30), histtype='step', label=kind)
    axes[2].hist(g.fill, bins=20, histtype='step', label=kind)
axes[0].set_xlabel('box size [px]'); axes[1].set_xlabel('distance [m]'); axes[2].set_xlabel('visible fill of the box')
for ax in axes: ax.legend()

In [ ]:
frames = pd.DataFrame([{'split': r['split'], 'night': r['night'], 'robot': r['robot_id'], 'mode': r['view_mode'],
    'target': r['target_kind'], 'objects': len(r['objects']), 'ignored': len(r['ignored'])} for r in records])
print(frames.groupby('split')[['objects', 'ignored']].mean())
print(frames.groupby(['mode', 'target'], dropna=False).size())
print(frames.groupby('robot').size(), frames.night.mean())

In [ ]:
r = next(r for r in records if len(r['objects']) >= 3)
image = cv2.cvtColor(cv2.imread(str(DATA/r['image'])), cv2.COLOR_BGR2RGB)
for o in r['objects']:
    x0, y0, x1, y1 = o['bbox_xyxy']; cv2.rectangle(image, (x0, y0), (x1-1, y1-1), (255, 255, 0), 2)
plt.figure(figsize=(9, 7)); plt.imshow(image); plt.title(r['image']); plt.axis('off')